In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# CELLULE 1 - Installation
import subprocess, sys
packages = "huggingface_hub bitsandbytes transformers peft accelerate trl datasets sentencepiece"
subprocess.run(f"{sys.executable} -m pip install -q -U {packages}", shell=True)
print("✅ Installation terminée")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.5 MB/s eta 0:00:00
✅ Installation terminée


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
# CELLULE 2 - Authentification
from huggingface_hub import login
import getpass
token = getpass.getpass("Token HF : ")
login(token=token)

Token HF :  ········


In [ ]:
import random
import numpy as np
import torch

SEED = 42

def fixer_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

fixer_seed()
print(f"Seed fixée à {SEED} pour reproductibilité")

In [3]:
# CELLULE 3 - Modèle SFT + LoRA trainable, float32, GPU unique
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_BASE = "Qwen/Qwen3-1.7B-Base"
MODEL_SFT = "UserMarrakech/qwen3-triage-medical"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32, bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE, quantization_config=bnb_config, dtype=torch.float32, device_map={"": 0},
)

model = PeftModel.from_pretrained(base_model, MODEL_SFT, is_trainable=True)
model.config.use_cache = False

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

print("Dtype LoRA :", set(p.dtype for n, p in model.named_parameters() if p.requires_grad))
print(f"Mémoire GPU après chargement : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 69.8MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Dtype LoRA : {torch.float32}
Mémoire GPU après chargement : 2.04 GB


In [4]:
# CELLULE 4 - Dataset DPO
from datasets import load_dataset
dataset_dpo = load_dataset("UserMarrakech/chsa-triage-dpo")
print(dataset_dpo)

README.md:   0%|          | 0.00/570 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.38MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  387kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/150 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected'],
        num_rows: 1350
    })
    validation: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected'],
        num_rows: 150
    })
})


In [5]:
# CELLULE 5 - Test DPO, batch=1, max_length=512
from trl import DPOConfig, DPOTrainer

dpo_config_test = DPOConfig(
    output_dir="/kaggle/working/dpo_test",
    max_steps=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    beta=0.1,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=5,
    eval_strategy="no",
    save_strategy="no",
    fp16=False,
    bf16=False,
    report_to="none",
    max_length=512,
    seed=SEED,
    data_seed=SEED,
)

trainer_dpo_test = DPOTrainer(
    model=model, args=dpo_config_test,
    train_dataset=dataset_dpo["train"], processing_class=tokenizer,
)

print("🚀 Test DPO (kernel propre, batch=1, max_length=512)...\n")
trainer_dpo_test.train()

Dropping fully truncated examples from train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Test DPO (kernel propre, batch=1, max_length=512)...



Step,Training Loss
5,0.678998
10,0.656833


TrainOutput(global_step=10, training_loss=0.6679152250289917, metrics={'train_runtime': 794.9313, 'train_samples_per_second': 0.201, 'train_steps_per_second': 0.013, 'total_flos': 1397041623859200.0, 'train_loss': 0.6679152250289917, 'epoch': 0.11922503725782414})

In [6]:
from trl import DPOConfig, DPOTrainer

REPO_MODELE_DPO = "UserMarrakech/qwen3-triage-medical-dpo"

dpo_config = DPOConfig(
    output_dir="/kaggle/working/dpo_final",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    beta=0.1,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    per_device_eval_batch_size=1,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    report_to="none",
    max_length=512,
    push_to_hub=True,
    hub_model_id=REPO_MODELE_DPO,
    hub_strategy="every_save",
    hub_private_repo=True,
)

trainer_dpo = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dataset_dpo["train"],
    eval_dataset=dataset_dpo["validation"],
    processing_class=tokenizer,
)

print("🚀 Lancement du run DPO complet (1 epoch, float32 stable)...\n")
trainer_dpo.train()

print("\n✅ Entraînement DPO terminé.")
trainer_dpo.push_to_hub(commit_message="Modèle final DPO")
print(f"Modèle poussé sur https://huggingface.co/{REPO_MODELE_DPO}")

Dropping fully truncated examples from train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized pr

Dropping fully truncated examples from eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

🚀 Lancement du run DPO complet (1 epoch, float32 stable)...



Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
50,0.609073,0.527861,0.988739,693856.000000,4.644702,4.710678,0.722498,-1.623068,-2.771025,0.753333,1.147957,-327.248393,-292.378553
84,0.542196,0.522109,0.988210,1170235.000000,4.712768,4.776658,0.722703,-1.535368,-2.750566,0.760000,1.215198,-326.371402,-292.173971



✅ Entraînement DPO terminé.
Modèle poussé sur https://huggingface.co/UserMarrakech/qwen3-triage-medical-dpo


In [7]:
from huggingface_hub import HfApi
api = HfApi()

# Cette ligne envoie directement votre dossier d'entraînement sur votre profil
api.upload_folder(
    folder_path="/kaggle/working/dpo_final",
    repo_id="UserMarrakech/qwen3-triage-medical-dpo",
    repo_type="model"
)
print("✅ Vos poids DPO ont été poussés manuellement avec succès !")


✅ Vos poids DPO ont été poussés manuellement avec succès !


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_DPO_ID = "UserMarrakech/qwen3-triage-medical-dpo"

print("📥 Chargement du modèle aligné par DPO depuis Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DPO_ID)

# Configuration de base 4-bit pour l'inférence rapide sur vos T4
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B-Base",
    quantization_config=bnb_config,
    device_map="auto"
)

# Fusionner l'adaptateur LoRA optimisé par DPO
model_dpo = PeftModel.from_pretrained(base_model, MODEL_DPO_ID)
model_dpo.eval()
print("✅ Modèle de triage DPO prêt pour l'évaluation !")

# Fonction d'inférence sécurisée (Anti-hallucination)
def simuler_triage_dpo(contexte, question):
    prompt = f"### Cas clinique :\n{contexte}\n\n### Question :\n{question}\n\n### Réponse :\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model_dpo.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.1,        # Température basse pour une rigueur clinique maximale
            repetition_penalty=1.2, # Bloque les répétitions de balises types <PERSON>
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    reponse_complete = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return reponse_complete.split("### Réponse :\n")[-1]

# --- TEST CLINIQUE COMPLEXE ---
print("\n--- 🧪 TEST DU MODÈLE DPO EN CONDITIONS RÉELLES ---")
cas_evaluation = (
    "Femme de 34 ans, sans antécédents, se présente pour une céphalée brutale "
    "et explosive apparue il y a 2 heures lors d'un effort (décrite comme un 'coup de tonnerre'). "
    "Elle présente des nausées et une légère raideur de nuque à l'examen. Constantes stables."
)
question_evaluation = "Proposer une analyse de triage, déterminer le niveau de priorité et l'orientation requise."

print(f"\n📝 Cas : {cas_evaluation}")
print("\n🤖 Décision du modèle DPO :")
print(simuler_triage_dpo(cas_evaluation, question_evaluation))


📥 Chargement du modèle aligné par DPO depuis Hugging Face...


tokenizer_config.json:   0%|          | 0.00/697 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/4.12k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 34.9MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

✅ Modèle de triage DPO prêt pour l'évaluation !

--- 🧪 TEST DU MODÈLE DPO EN CONDITIONS RÉELLES ---

📝 Cas : Femme de 34 ans, sans antécédents, se présente pour une céphalée brutale et explosive apparue il y a 2 heures lors d'un effort (décrite comme un 'coup de tonnerre'). Elle présente des nausées et une légère raideur de nuque à l'examen. Constantes stables.

🤖 Décision du modèle DPO :
- Analyse biologique : 
    - Hémogramme avec hématies totales
     <PERSON> sanguin complet incluant plaquettes,
      urines récentes en quantités libres ou formelles.
       SgErythrocytiques normales chez la femme jeune ; hétérogénéité du sang normal dans les cas anormaux évoqués ici mais pas systématiquement recherché par défaut ;
        Plaquettes augmentées souvent associées aux autres anomalies métaboliques mentionnées ci-dessous;
         Thrombopénie sérique typique au-delà de quelques centaines/mm³ si thromboembolie pulmonaire suspecte; rarement causée directement par infection virale même